In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification, DistilBertModel, TextClassificationPipeline
from datasets import load_dataset
from graphpatch import PatchableGraph, ZeroPatch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm, trange

In [9]:
# Load model and tokenizer
model_name = "martin-ha/toxic-comment-model"
model = DistilBertModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load dataset
dataset = load_dataset("toxigen/toxigen-data", "train")["train"]

In [117]:
# Function to evaluate model on dataset
def evaluate_model(model, dataset, tokenizer):
    model.eval()
    total_loss = 0
    criterion = torch.nn.CrossEntropyLoss()

    for i in trange(1000):
        input_text = dataset["generation"][i]
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits[0]
        label = dataset["prompt_label"][i]
        labels = torch.zeros_like(logits)
        labels[label] = 1
        loss = criterion(logits, labels)
        total_loss += loss.item()

    return total_loss / len(dataset)

# Baseline evaluation
# baseline_loss = evaluate_model(model, dataset, tokenizer)
# print(f"Baseline loss: {baseline_loss:.4f}")

# Function to ablate an attention head using ZeroPatch
def ablate_attention_head(model, layer_idx, head_idx):
    with PatchableGraph(model) as graph:
        attention_module = model.transformer.h[layer_idx].attn

        # Create a TensorSlice targeting the specific attention head
        slice_target = TensorSlice[:, head_idx, :, :]  # Selects the head's output in multi-head attention

        # Apply ZeroPatch
        patch = ZeroPatch(slice=slice_target)
        graph[attention_module, "attn_output"].patch(patch)

        # Evaluate the patched model
        ablated_loss = evaluate_model(model, dataset, tokenizer)

    return ablated_loss

# Iterate over attention heads and evaluate impact
cfg = model.config
num_layers = cfg.n_layers
num_heads = cfg.n_heads

impact_scores = np.zeros((num_layers, num_heads))

for layer in range(num_layers):
    for head in range(num_heads):
        ablated_loss = ablate_attention_head(model, layer, head)
        impact_scores[layer, head] = ablated_loss - baseline_loss
        print(f"Layer {layer}, Head {head} - Loss Difference: {impact_scores[layer, head]:.4f}")

# Save results
np.save("attention_head_impact.npy", impact_scores)
print("Ablation impact scores saved.")


/home/kokil/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:4664: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(


ValueError: You have to specify either input_ids or inputs_embeds

In [10]:
input_text = dataset["generation"][0]
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)

In [11]:
pg = PatchableGraph(model, **inputs, use_cache=False)

/home/kokil/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:4779: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(


TypeError: forward() got an unexpected keyword argument 'use_cache'

In [116]:
model.config

DistilBertConfig {
  "_name_or_path": "martin-ha/toxic-comment-model",
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "id2label": {
    "0": "non-toxic",
    "1": "toxic"
  },
  "initializer_range": 0.02,
  "label2id": {
    "non-toxic": 0,
    "toxic": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "torch_dtype": "float32",
  "transformers_version": "4.43.3",
  "vocab_size": 30522
}

In [115]:
dir(model)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_assisted_decoding',
 '_auto_class',
 '_autoset_attn_implementation',
 '_backward_compatibility_gradient_checkpointing',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_beam_search',
 '_buffers',
 '_call_impl',
 '_check_and_enable_flash_attn_2',
 '_check_and_enable_sdpa',
 '_compiled_call_impl',
 '_constrained_beam_search',
 '_contrastive_search',
 '_convert_head_mask_to_5d',
 '_copy_lm_head_original_to_resized',
 '_create_repo',
 '_dispatch_accelerate_model',
 '_dola_decoding',
 '_expand_inputs_for_generation',